# Exercise 2.4: (Modern C++) Ranges

From *Programming in High Energy Particle Physics*, Chapter 2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch02/ex02_4.ipynb)

Exercise 2.4 (Modern C++) Ranges Rewrite the filter function from Exercise 2.2 using C++20 ranges ( std::views::filter ) instead of the STL std::copy_if . Compare the readability and performance of both approaches. Profile the execution time for 100,000 elements using std::chrono . Compile with -std=c++20 (GCC ≥ 10 or Clang ≥ 15 for full ranges support) and with optimization ( -O2 ) enabled.

<details><summary>Hint</summary>

auto view = jets | std::views::filter(pred); is lazy: nothing is copied until you iterate. To compare fairly with copy_if , either materialize the view into a vector or time a loop that consumes both results. Use std::chrono::steady_clock::now() before and after, and repeat the measurement several times.

</details>

In [ ]:
import subprocess, shutil, sys
IN_COLAB = "google.colab" in sys.modules
assert shutil.which("g++"), "A C++ compiler (g++) is required"

### Compare eager copying and a materialized lazy view

Five repeats reduce timing noise. Both methods must produce the same output.

In [ ]:
from pathlib import Path
source = r'''#include <algorithm>
#include <chrono>
#include <iostream>
#include <iterator>
#include <ranges>
#include <vector>
int main(){std::vector<int> jets(100000);
  for(size_t i=0;i<jets.size();++i) jets[i]=i%100;
  auto pred=[](int pt){return pt>30;};
  using clock=std::chrono::steady_clock;
  double copy_ms=0,range_ms=0; size_t n_copy=0,n_range=0;
  for(int trial=0;trial<5;++trial){
    auto t0=clock::now(); std::vector<int> copied;
    std::copy_if(jets.begin(),jets.end(),std::back_inserter(copied),pred);
    auto t1=clock::now(); std::vector<int> materialized;
    // TODO: materialize jets | std::views::filter(pred).
    auto t2=clock::now();
    n_copy=copied.size();n_range=materialized.size();
    copy_ms+=std::chrono::duration<double,std::milli>(t1-t0).count();
    range_ms+=std::chrono::duration<double,std::milli>(t2-t1).count();
  }
  std::cout << "copy_if=" << copy_ms/5 << " ms ranges=" << range_ms/5
            << " ms speedup(copy/ranges)=" << copy_ms/range_ms
            << " selected=" << n_range << "\n";
  return 0;
}'''
Path("exercise.cpp").write_text(source)
build = subprocess.run(["g++", "-O2", "-std=c++20", "exercise.cpp", "-o", "exercise"], capture_output=True, text=True)
print(build.stderr)
assert build.returncode == 0
run = subprocess.run(["./exercise"], capture_output=True, text=True, check=True)
print(run.stdout)

In [ ]:
print("Selected counts should agree after the TODO is filled")